# MiniGPT-Stack Phase 6 / S4 - layer-4 SAE FULL RUN (A1-A4 acceptance measurement)

Rules: `docs/phase6_preregistration.md` (thresholds unchanged, incl. Amendment 1: window position 0 is
excluded from SAE training). The probe that preceded this run is recorded in `docs/decisions.md`.

- **192 epochs** of 523,264 fresh training activations (100.5M tokens), a new buffer every epoch, checkpoint every 5.
- **Training limit 4.5 h** (`MAX_HOURS`). If the limit is hit or fewer than 192 epochs finish, the report is written
  as TRUNCATED and the kernel **raises**, so it ends as an ERROR and never as COMPLETE. A hard-cap watchdog exits
  the process at 5.75 h (also an ERROR).
- SHA-256 gate on the base checkpoint; fast tests are fatal on failure. A5 (SFT transfer), layers 2 and 6, and
  labeling are NOT part of this run. COMPLETE means "finished 192/192 epochs and measured A1-A4"; whether the
  layer passes is in the printed verdict.

**Attach two datasets:** `glassbox-sft-init-weights` and `tinystories-shards-8192`.


In [ ]:
# ---- PINNED (see docs/phase6_preregistration.md and docs/phase6_s4_full_run.md) ----
CONFIG        = "configs/base_30m.yaml"
DEVICE        = "cuda"
LAYER         = 4
N_FEATURES    = 4096
K             = 32
LR            = 4e-4
WARMUP        = 50
BATCH_TOKENS  = 4096
BUFFER_TOKENS = 524_288
EPOCHS        = 192           # 192 x 523,264 rows = 100.5M training tokens (pre-registered ~100M)
CTX           = 512
SEED          = 0
NORM_WINDOWS  = 256
EVAL_WINDOWS  = 64            # per-epoch quick eval (held-out validation windows)
EVAL_EVERY    = 5             # epochs
CKPT_EVERY    = 5             # epochs (atomic, resume-exact)
MAX_HOURS     = 4.5           # training wall-clock limit; hitting it = TRUNCATED = kernel ERROR
EVAL_TOKENS   = 500_000       # A1-A4 evaluation tokens (pre-registered, unchanged)
DEAD_TOKENS   = 10_000_000    # A3 scan (pre-registered, unchanged; never reduced)

WEIGHTS_FILE   = "weights_step12158.pt"
WEIGHTS_BYTES  = 125_893_589
WEIGHTS_SHA256 = "feb065a5bc7d0ca240d756cd7bc48d8e1e284349a35dcd783ece4dc0c899aab0"

import faulthandler, os, sys, threading, time
T0 = time.time()
HARD_CAP_S = int(5.75 * 3600)   # 4.5 h training + sweep/setup + 10M-token eval + margin; the process exits here
COMPLETED = False
_WATCHDOG_TIMER = None


def arm_watchdog(seconds):
    """Hard cap: the kernel PROCESS exits at the cap, so the run ends as an ERROR (dead kernel)
    and never as COMPLETE. faulthandler needs a real file descriptor; ipykernel's sys.stderr
    has none (io.UnsupportedOperation: fileno), so the original stderr is used. If that fails
    too, a timer thread calls os._exit."""
    global _WATCHDOG_TIMER
    try:
        faulthandler.dump_traceback_later(seconds, exit=True, file=sys.__stderr__)
        return "faulthandler"
    except Exception:                                           # noqa: BLE001
        def die():
            print(f"[full] HARD CAP {seconds}s reached: exiting the kernel", file=sys.__stderr__, flush=True)
            os._exit(3)
        _WATCHDOG_TIMER = threading.Timer(seconds, die)
        _WATCHDOG_TIMER.daemon = True
        _WATCHDOG_TIMER.start()
        return "timer"


def disarm_watchdog():
    faulthandler.cancel_dump_traceback_later()
    if _WATCHDOG_TIMER is not None:
        _WATCHDOG_TIMER.cancel()


WATCHDOG = arm_watchdog(HARD_CAP_S)
print("[full] hard-cap watchdog armed via", WATCHDOG, "at", HARD_CAP_S, "s", flush=True)


def time_left():
    return HARD_CAP_S - (time.time() - T0)


def capped(argv):
    import subprocess
    left = time_left()
    if left < 30:
        raise RuntimeError(f"[full] hard cap: only {left:.0f}s left; not starting {argv[:4]}")
    return subprocess.run(argv, timeout=left)


## GPU check

In [ ]:
import torch
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("torch", torch.__version__, "| cuda build", torch.version.cuda, "| cuda avail", torch.cuda.is_available())
assert torch.cuda.is_available(), "enable GPU (T4) in notebook settings"


## Clone

In [ ]:
import os
REPO = "https://github.com/Priyanshaur/Glassbox.git"
ROOT = "/kaggle/working/Glassbox"
if not os.path.exists(ROOT):
    !git clone $REPO $ROOT
else:
    !git -C $ROOT pull --ff-only
%cd $ROOT
!git rev-parse --short HEAD
import inspect
from model.gpt import GPT as _G
assert "stop_after" in inspect.signature(_G.forward).parameters, "clone lacks the early-exit commit"
COMMIT = os.popen("git rev-parse --short HEAD").read().strip()
print("[full] repo commit", COMMIT, "| early-exit (stop_after) present", flush=True)


## Install

In [ ]:
# Kaggle already ships numpy, pyyaml, tqdm, pytest, regex and datasets.
!pip install -q regex pyyaml pytest tqdm datasets


## Inputs - base checkpoint, byte-exact and SHA-256 gate

In [ ]:
import glob, hashlib, os, torch

hits = glob.glob("/kaggle/input/**/" + WEIGHTS_FILE, recursive=True)
assert hits, "attach the private dataset glassbox-sft-init-weights (contains weights_step12158.pt)"
W = hits[0]
got = os.path.getsize(W)
print(W, "| bytes", got, "(expected", WEIGHTS_BYTES, ")")
assert got == WEIGHTS_BYTES, "weights size mismatch (truncated upload?)"
h = hashlib.sha256()
with open(W, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print("[full] sha256 expected  ", WEIGHTS_SHA256)
print("[full] sha256 in-kernel ", h.hexdigest(), "->", "MATCH" if h.hexdigest() == WEIGHTS_SHA256 else "MISMATCH")
assert h.hexdigest() == WEIGHTS_SHA256, "SHA-256 mismatch; stopping before anything runs"
SHA_OK = True
SHA_SEEN = h.hexdigest()
print("[full] weights file = base model weights_step12158.pt (pretraining final step); sha256", SHA_SEEN, flush=True)

from common.config import load_config
from model.gpt import GPT, count_params
cfg = load_config(CONFIG)
model = GPT(cfg["model"])
model.load_state_dict(torch.load(W, map_location="cpu", weights_only=True))
model = model.to(DEVICE).eval()
print("base weights load into the configured arch | params:", count_params(model), "| device", DEVICE)


## Shards (real train + validation)

In [ ]:
import glob, json, os
hits = glob.glob("/kaggle/input/**/meta.json", recursive=True)
assert hits, "attach the shards dataset first"
print("meta.json candidates:", hits)
shard_dir = os.path.dirname(hits[0])
print("--- meta.json head ---")
print("\n".join(open(os.path.join(shard_dir, "meta.json")).read().splitlines()[:15]))
os.makedirs("data", exist_ok=True)
# The cloned repo can contain a stale data/meta.json (smoke-test counts). If it is left in
# place, the symlink below is skipped and training would read the wrong token counts.
for f in glob.glob("data/*.bin") + ["data/meta.json"]:
    if os.path.lexists(f):
        os.remove(f)
for f in glob.glob(os.path.join(shard_dir, "*.bin")) + [os.path.join(shard_dir, "meta.json")]:
    os.symlink(f, os.path.join("data", os.path.basename(f)))
print("linked:", sorted(os.listdir("data")))
meta = json.load(open("data/meta.json"))
assert "shards" in meta and "vocab_size" in meta, meta.keys()
for s in meta["shards"]:
    assert {"split", "index", "file", "tokens"} <= set(s), s
print("meta keys OK:", sorted(meta.keys()))
tr = sum(s["tokens"] for s in meta["shards"] if s["split"] == "train")
va = sum(s["tokens"] for s in meta["shards"] if s["split"] == "validation")
print(f"EXACT counts from data/meta.json: train={tr} val={va} vocab={meta['vocab_size']}")
for s in meta["shards"]:
    nbytes = os.path.getsize(os.path.join("data", s["file"]))
    print(f"  {s['file']} {s['tokens']} toks {nbytes/1024/1024:.1f}MB")
    assert nbytes == 2 * s["tokens"], f"{s['file']}: file size does not match meta (stale meta.json?)"
assert tr > 400_000_000, f"train tokens {tr} look wrong (expected ~467M)"
TRAIN_TOKS, VAL_TOKS = tr, va

## Fast tests - fatal on failure

In [ ]:
import subprocess, sys

proc = capped([sys.executable, "-m", "pytest", "tests/test_hooks.py", "tests/test_model.py",
               "tests/test_interp_activations.py", "tests/test_interp_sae.py", "tests/test_interp_train.py",
               "tests/test_interp_eval.py", "tests/test_early_exit.py", "-q"])
if proc.returncode != 0:
    raise RuntimeError(f"[full] fast tests FAILED (rc={proc.returncode}); stopping")
TESTS_OK = True
print("[full] fast tests: PASS", f"| {time.time() - T0:.0f}s elapsed", flush=True)


## Scale, norm stats, extraction speed (with the early-exit forward)

Same measurements as the probe, for comparison: the probe's extraction rates were 40,148 / 35,963 / 36,839 tok/s
(chunk 32 / 64 / 128) with the full forward.


In [ ]:
import json, numpy as np
from interp import activations as A
from train.data_loader import ShardLoader

tr = ShardLoader(cfg["data"]["out_dir"], "train")
va = ShardLoader(cfg["data"]["out_dir"], "validation")
RUN = {"layer": LAYER, "label": "layer-4 full run", "commit": COMMIT}

SCALE = A.estimate_norm_scale(model, tr, LAYER, SEED, NORM_WINDOWS, CTX, 64, DEVICE)
stats = A.norm_stats(model, tr, LAYER, SEED, NORM_WINDOWS, CTX, 64, DEVICE)
RUN["scale"] = SCALE
RUN["norm_stats"] = stats
print(f"[full] scale (real windows, positions >= 1) = {SCALE:.5f}")
print("[full] layer-4 norms:", {k: round(v, 2) for k, v in stats.items() if "norm" in k or "max" in k})


def timed_fill(chunk):
    torch.cuda.synchronize(); t = time.perf_counter()
    A.fill_buffer(model, tr, LAYER, SEED, 12345, BUFFER_TOKENS, CTX, SCALE, chunk, DEVICE)
    torch.cuda.synchronize()
    return time.perf_counter() - t


ext = {}
rows = A.buffer_rows(BUFFER_TOKENS, CTX)
for chunk in (32, 64, 128):
    timed_fill(chunk)                                           # warm-up
    ts = sorted(timed_fill(chunk) for _ in range(3))
    ext[chunk] = {"median_s": ts[1], "tokens_per_s": rows / ts[1]}
    print(f"[full] extraction chunk={chunk:>3}: {ts[1]:.2f}s per {rows:,} rows -> {rows / ts[1]:,.0f} tok/s")
RUN["extraction"] = ext
BEST_CHUNK = max(ext, key=lambda c: ext[c]["tokens_per_s"])
print("[full] best extraction chunk:", BEST_CHUNK, f"| {time.time() - T0:.0f}s elapsed", flush=True)


## Training - 192 epochs, checkpoint every 5, exact resume

`train()` resumes from `ckpt.pt` if one exists in the output directory (or, for a continuation session, a
checkpoint attached from a dataset is copied there first). A run that stops for any reason other than finishing
192/192 epochs is written up as TRUNCATED and the kernel raises.


In [ ]:
import glob, json, shutil, numpy as np
from interp.train_sae import TrainConfig, train

OUT = "/kaggle/working/sae_full"
os.makedirs(OUT, exist_ok=True)
prev = glob.glob("/kaggle/input/**/sae_full/ckpt.pt", recursive=True)
if prev and not os.path.exists(OUT + "/ckpt.pt"):
    shutil.copy(prev[0], OUT + "/ckpt.pt")
    print("[full] continuing from attached checkpoint", prev[0], flush=True)

room_h = (time_left() - 25 * 60) / 3600
assert room_h > 0.25, f"not enough time left for a training stage: {room_h:.2f} h"
budget_h = min(MAX_HOURS, room_h)
tcfg = TrainConfig(layer=LAYER, n_features=N_FEATURES, k=K, seed=SEED, lr=LR, warmup_steps=WARMUP,
                   batch_tokens=BATCH_TOKENS, buffer_tokens=BUFFER_TOKENS, epochs=EPOCHS, ctx=CTX,
                   norm_windows=NORM_WINDOWS, chunk=BEST_CHUNK, log_every=200, eval_every=EVAL_EVERY,
                   ckpt_every=CKPT_EVERY, eval_windows=EVAL_WINDOWS, max_hours=budget_h)
print(f"[full] training budget {budget_h:.2f} h, {EPOCHS} epochs of {BUFFER_TOKENS:,} tokens, "
      f"chunk {BEST_CHUNK}, ckpt every {CKPT_EVERY}", flush=True)
t_tr = time.time()
res = train(model, tr, va, tcfg, OUT, device=DEVICE, log=print,
            meta_extra={"label": "layer-4 full run", "weights_sha256": WEIGHTS_SHA256, "commit": COMMIT})
TRAIN_WALL = time.time() - t_tr
rows_m = [json.loads(l) for l in open(OUT + "/metrics_sae.jsonl", encoding="utf-8") if l.strip()]
evals = [(r["epoch"] + 1, r["eval"]) for r in rows_m if "eval" in r]
stats_e = [r["epoch_stats"] for r in rows_m if "epoch_stats" in r]
term = [r for r in rows_m if "terminal" in r][-1]
EPOCHS_DONE = int(term["epochs_done"])
med = lambda xs: float(np.median(xs)) if xs else float("nan")
RUN["training"] = {
    "epochs_planned": EPOCHS, "epochs_done": EPOCHS_DONE, "terminal": term["terminal"], "wall_s": TRAIN_WALL,
    "fill_s_median": med([s["fill_s"] for s in stats_e]), "train_s_median": med([s["train_s"] for s in stats_e]),
    "epoch_s_median": med([s["fill_s"] + s["train_s"] for s in stats_e]),
    "train_tokens_per_s_median": med([s["train_tokens_per_s"] for s in stats_e]),
    "extract_tokens_per_s_median": med([s["extract_tokens_per_s"] for s in stats_e]),
    "peak_mem_mb": max([s.get("peak_mem_mb", 0) for s in stats_e], default=0),
    "fvu_trajectory": [{"epoch": e, **ev} for e, ev in evals]}
for e, ev in evals:
    print(f"[full] epoch {e:>3}: fvu {ev['fvu']:.4f} (excl pos0 {ev.get('fvu_excl_pos0', float('nan')):.4f}) "
          f"l0 {ev['l0']:.1f} alive {ev['alive_frac']:.3f}")
print(f"[full] trained {EPOCHS_DONE}/{EPOCHS} epochs in {TRAIN_WALL / 3600:.2f} h, terminal={term['terminal']}", flush=True)

TRUNCATED = (term["terminal"] != "epochs") or (EPOCHS_DONE != EPOCHS)
if TRUNCATED:
    RUN["status"] = "TRUNCATED"
    RUN["kernel_elapsed_s"] = time.time() - T0
    json.dump(RUN, open("/kaggle/working/full_report.json", "w"), indent=1, default=float)
    raise RuntimeError(f"[full] TRUNCATED: {EPOCHS_DONE}/{EPOCHS} epochs (terminal={term['terminal']}); "
                       f"checkpoint kept in {OUT}; the evaluation is NOT run and this is NOT COMPLETE.")


## A1-A4 on the finished SAE (pre-registered thresholds, unchanged)

500,000 evaluation tokens, the full 10M-token dead-feature scan (never reduced), CE recovered with the
mean-ablation baseline over the full forward, plus the position-0 extras of Amendment 1. A5 is not evaluated.


In [ ]:
from interp import eval_sae as E
from interp.sae import load_sae
from tokenizer.bpe import BPETokenizer

if time_left() < 20 * 60:
    raise RuntimeError(f"[full] only {time_left():.0f}s left: not enough for the full evaluation; not reducing it")
sae, meta = load_sae(OUT + "/sae_latest.pt")
assert meta["epochs_done"] == EPOCHS, meta
sae = sae.to(DEVICE).eval()
torch.cuda.synchronize(); t_ev = time.time()
rep = E.evaluate(model, sae, meta["act_scale"], LAYER, va, tr, CTX,
                 BPETokenizer.load(cfg["data"]["tokenizer_path"]), cfg["data"]["out_dir"],
                 eval_tokens=EVAL_TOKENS, dead_tokens=DEAD_TOKENS, sft_model=None, chunk=32)
torch.cuda.synchronize()
EVAL_S = time.time() - t_ev
rep["eval_seconds"] = EVAL_S
RUN["a1_a4"] = rep
print(f"[full] A1 fvu {rep['fvu']:.4f} (<= 0.15) | A2 l0 {rep['l0']:.2f} (8..32) | A3 dead {rep['dead_fraction']:.3f} "
      f"(<= 0.10, {rep['scan_tokens']:,} tokens) | A4 ce recovered {rep['ce_recovered']:.3f} (>= 0.85) | eval {EVAL_S:.0f}s")
print("[full] fvu by position:", {k: round(v, 4) for k, v in rep["fvu_by_position"].items()})
print("[full] excl pos0:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in rep["excl_pos0"].items()})
print("[full] VERDICT A1-A4:", rep["verdict"], "| excl pos0:", rep["verdict_excl_pos0"], flush=True)


## Report and COMPLETE guard

In [ ]:
RUN["status"] = "FINISHED_192_EPOCHS"
RUN["kernel_elapsed_s"] = time.time() - T0
RUN["hash_gate"] = "PASS" if SHA_OK else "FAIL"
RUN["weights_sha256"] = SHA_SEEN
RUN["watchdog"] = WATCHDOG
RUN["tests"] = "PASS" if TESTS_OK else "FAIL"
RUN["a5"] = "not evaluated (not part of this run)"
json.dump(RUN, open("/kaggle/working/full_report.json", "w"), indent=1, default=float)
t_, v_ = RUN["training"], RUN["a1_a4"]["verdict"]
lines = ["# Phase 6 S4 layer-4 full run summary", "",
         f"- commit {COMMIT}; hash gate {RUN['hash_gate']}; fast tests {RUN['tests']}; kernel elapsed {RUN['kernel_elapsed_s'] / 3600:.2f} h",
         f"- training: {t_['epochs_done']}/{t_['epochs_planned']} epochs in {t_['wall_s'] / 3600:.2f} h, epoch median {t_['epoch_s_median']:.1f}s "
         f"(fill {t_['fill_s_median']:.1f}s + train {t_['train_s_median']:.1f}s), peak mem {t_['peak_mem_mb']:.0f} MB",
         f"- A1 fvu {RUN['a1_a4']['fvu']:.4f} | A2 l0 {RUN['a1_a4']['l0']:.2f} | A3 dead {RUN['a1_a4']['dead_fraction']:.4f} | A4 ce recovered {RUN['a1_a4']['ce_recovered']:.4f}",
         f"- verdict flags: {v_}"]
open("/kaggle/working/full_summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
COMPLETED = True
disarm_watchdog()


In [ ]:
if COMPLETED:
    print("=" * 78)
    print(f"[full] COMPLETE: {EPOCHS}/{EPOCHS} epochs trained and A1-A4 measured (A5 not run). The verdict is the line printed above.")
    print("=" * 78, flush=True)
else:
    raise RuntimeError("[full] COMPLETED sentinel is False; NOT reporting COMPLETE.")
